In [33]:
import os
import pickle

import torch
from torch import nn

from skimage.io import imread
from skimage.transform import resize
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score

# prepare data
input_dir = 'C:/Users/olive/OneDrive/Documents/Tomato Dataset/PlantVillage'
categories = ['Tomato__Target_Spot', 'Tomato__Tomato_mosaic_virus', 'Tomato__Tomato_YellowLeaf__Curl_Virus', 'Tomato_Bacterial_spot', 'Tomato_Early_blight', 'Tomato_healthy', 'Tomato_Late_blight', 'Tomato_Leaf_Mold', 'Tomato_Septoria_leaf_spot', 'Tomato_Spider_mites_Two_spotted_spider_mite']
IMAGE_EXTENSIONS = ('.jpg', '.jpeg', '.png')

data = []
labels = []
for category_idx, category in enumerate(categories):
    for file in os.listdir(os.path.join(input_dir, category)):
        if not file.lower().endswith(IMAGE_EXTENSIONS):
            continue
        img_path = os.path.join(input_dir, category, file)
        img = imread(img_path)
        img = resize(img, (32, 32))
        data.append(img)
        labels.append(category_idx)

data = np.asarray(data)
labels = np.asarray(labels)

In [34]:
# train / test split
x_train, x_test, y_train, y_test = train_test_split(data, labels, test_size=0.2, shuffle=True, stratify=labels)
print(x_train.shape, y_train.shape)

(12808, 32, 32, 3) (12808,)


In [35]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cpu device


In [36]:
from torch import nn

class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(15 * 15 * 3, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

        self.conv_stack = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 2 * 2, 256),
            nn.ReLU(),
            nn.Linear(256, 10),
        )

    def forward(self, x):
        return self.classifier(self.conv_stack(x))
    
model = NeuralNetwork().to(device)
print(model)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=675, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
  (conv_stack): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=512, out_features=256, bias=True)
    (2): ReLU()
    (3): Linear(in_feature

In [37]:
from torch.utils.data import TensorDataset, DataLoader

X_train = torch.tensor(x_train, dtype=torch.float32).permute(0, 3, 1, 2)
y_train_t = torch.tensor(y_train, dtype=torch.long)
X_test = torch.tensor(x_test, dtype=torch.float32).permute(0, 3, 1, 2)
y_test_t = torch.tensor(y_test, dtype=torch.long)

train_dataset = TensorDataset(X_train, y_train_t)
test_dataset = TensorDataset(X_test, y_test_t)

train_dataloader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=64)


In [ ]:
learning_rate = 1e-3
batch_size = 64
epochs = 150

def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    # Set the model to training mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        # Compute prediction and loss
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), batch * batch_size + len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")


def test_loop(dataloader, model, loss_fn):
    # Set the model to evaluation mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    # Evaluating the model with torch.no_grad() ensures that no gradients are computed during test mode
    # also serves to reduce unnecessary gradient computations and memory usage for tensors with requires_grad=True
    with torch.no_grad():
        for X, y in dataloader:
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

In [41]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)


for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loop(train_dataloader, model, loss_fn, optimizer)
    test_loop(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 2.248417  [   64/12808]
loss: 2.150137  [ 6464/12808]
loss: 1.833567  [12808/12808]
Test Error: 
 Accuracy: 20.0%, Avg loss: 2.194445 

Epoch 2
-------------------------------
loss: 2.250696  [   64/12808]
loss: 2.129503  [ 6464/12808]
loss: 2.267776  [12808/12808]
Test Error: 
 Accuracy: 20.0%, Avg loss: 2.183383 

Epoch 3
-------------------------------
loss: 2.243240  [   64/12808]
loss: 2.130269  [ 6464/12808]
loss: 2.321457  [12808/12808]
Test Error: 
 Accuracy: 20.0%, Avg loss: 2.154050 

Epoch 4
-------------------------------
loss: 2.208618  [   64/12808]
loss: 2.057280  [ 6464/12808]
loss: 2.281103  [12808/12808]
Test Error: 
 Accuracy: 27.7%, Avg loss: 2.025600 

Epoch 5
-------------------------------
loss: 2.168483  [   64/12808]
loss: 2.017086  [ 6464/12808]
loss: 2.031833  [12808/12808]
Test Error: 
 Accuracy: 30.3%, Avg loss: 1.891182 

Epoch 6
-------------------------------
loss: 1.813666  [   64/12808]
loss: 1.737034  [ 64

In [45]:
import torchvision.models as models

torch.save(model.state_dict(), 'model_weights.pth')
